# glofas2schism_shp2csv

Convert GloFAS/SCHISM source shapefiles to CSV files expected by `gen_source.py`.

## What this does

The `gen_source` workflow step expects two CSV files in `fix/`:

| File | Description |
|------|-------------|
| `fix/source_glofas.csv` | Points placed on GloFAS grid cells to extract discharge from |
| `fix/source_schism.csv` | Same rivers repositioned to their injection location inside the SCHISM mesh |

Both files must have columns: `id, lon, lat`

This notebook reads the two shapefiles you created in QGIS and writes the
corresponding CSVs. Run it once (or again if you update the shapefiles).

## Prerequisites

- `geopandas` must be installed in your Python environment
- The two shapefiles (`source_glofas.shp`, `source_schism.shp`) must exist
- Both shapefiles must have an `id` column (integer, unique per river)
- Points should be in WGS84 (EPSG:4326) or any CRS — they will be
  reprojected to WGS84 automatically

In [ ]:
import geopandas as gpd
import warnings


def extract_lon_lat(shp_filepath, csv_filepath, id_column_name='id'):
    """
    Read a shapefile, extract lon/lat coordinates, and save to a CSV.

    Parameters
    ----------
    shp_filepath : str
        Path to the input .shp file.
    csv_filepath : str
        Path for the output .csv file.
    id_column_name : str
        Exact name of the ID column in the shapefile (case-sensitive).
        Default: 'id'

    Output columns: id, lon, lat
    """
    print(f"Reading {shp_filepath}...")
    gdf = gpd.read_file(shp_filepath)

    # Validate ID column
    if id_column_name not in gdf.columns:
        raise ValueError(
            f"Column '{id_column_name}' not found. "
            f"Available columns: {list(gdf.columns)}"
        )

    # Reproject to WGS84 if needed
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        print("  Reprojecting to WGS84 (EPSG:4326)...")
        gdf = gdf.to_crs(epsg=4326)

    # Extract coordinates.
    # For Point geometries this gives the point coordinates directly.
    # For Polygon/Line geometries this gives the centroid.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        gdf['lon'] = gdf.geometry.centroid.x
        gdf['lat'] = gdf.geometry.centroid.y

    # Keep only id, lon, lat
    df_output = gdf[[id_column_name, 'lon', 'lat']].rename(
        columns={id_column_name: 'id'}
    )

    df_output.to_csv(csv_filepath, index=False)
    print(f"  {len(df_output)} rivers written to {csv_filepath}")
    print(df_output.head())

## Configuration

Set the paths to your shapefiles and the desired output CSV locations.

The output CSVs should be copied (or written directly) to `M01/fix/`.

In [ ]:
# --- Input shapefiles (edit these paths) ---
SHP_GLOFAS = r"/path/to/source_glofas.shp"
SHP_SCHISM = r"/path/to/source_schism.shp"

# --- Output CSVs (should match fix/ on Hercules) ---
CSV_GLOFAS = r"/path/to/fix/source_glofas.csv"
CSV_SCHISM = r"/path/to/fix/source_schism.csv"

# --- ID column name in your shapefiles ---
ID_COL = 'id'

## Run the conversion

In [ ]:
print("=== source_glofas ===")
extract_lon_lat(SHP_GLOFAS, CSV_GLOFAS, id_column_name=ID_COL)

print()
print("=== source_schism ===")
extract_lon_lat(SHP_SCHISM, CSV_SCHISM, id_column_name=ID_COL)

## Verify

Check that the two CSVs have the same IDs and the coordinates look sensible
for the Alaska domain (lon ~150–230, lat ~45–78).

In [ ]:
import pandas as pd

df_gf = pd.read_csv(CSV_GLOFAS)
df_sc = pd.read_csv(CSV_SCHISM)

print(f"source_glofas: {len(df_gf)} rivers")
print(df_gf.describe())

print(f"\nsource_schism: {len(df_sc)} rivers")
print(df_sc.describe())

# Check IDs match
missing_in_schism = set(df_gf['id']) - set(df_sc['id'])
missing_in_glofas = set(df_sc['id']) - set(df_gf['id'])
if missing_in_schism:
    print(f"\nWARNING: IDs in glofas but not schism: {sorted(missing_in_schism)}")
if missing_in_glofas:
    print(f"WARNING: IDs in schism but not glofas: {sorted(missing_in_glofas)}")
if not missing_in_schism and not missing_in_glofas:
    print("\nID check passed: both files have identical IDs.")